In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Cargar dataset completo de Starbucks
base_url = "https://raw.githubusercontent.com/WalterVillatoro1204/marketing-analytics-walter-villatoro/main/Semana_3"
df = pd.read_csv(f"{base_url}/uplift_experiment_FULL.csv")

FEATURES = ["recency_days", "frequency", "monetary", "email_open_rate"]

# Separar train y test (70% train, 30% test)
train, test = train_test_split(df, test_size=0.30, random_state=42, stratify=df["treatment_group"])
test = test.copy()

In [4]:
# 1. Variable objetivo: 1 si no respondio, 0 si respondio
y_train_churn = 1 - train["responded_90d"]

# 2. Entrenar regresion logistica
modelo_churn = LogisticRegression(max_iter=1000)
modelo_churn.fit(train[FEATURES], y_train_churn)

# 3. Predecir probabilidad de churn en test
test["prob_churn"] = modelo_churn.predict_proba(test[FEATURES])[:, 1]

# 4. Seleccionar el 30% con mayor riesgo
n_30 = int(len(test) * 0.30)
top_riesgo = test.sort_values("prob_churn", ascending=False).head(n_30)

# 5. Medir valor incremental real de este 30%
val_t = top_riesgo.loc[top_riesgo["treatment_group"] == "treatment", "value_90d"].mean()
val_c = top_riesgo.loc[top_riesgo["treatment_group"] == "control", "value_90d"].mean()
uplift_riesgo_val = val_t - val_c
total_riesgo = uplift_riesgo_val * n_30

print(f"--- Politica por Riesgo (30%) ---")
print(f"Uplift promedio por cliente: ${uplift_riesgo_val:.2f}")
print(f"Valor incremental total: ${total_riesgo:.2f}")

--- Politica por Riesgo (30%) ---
Uplift promedio por cliente: $0.17
Valor incremental total: $219.73


In [5]:
# 1. Separar train en tratados y control
train_t = train[train["treatment_group"] == "treatment"]
train_c = train[train["treatment_group"] == "control"]

# 2. Entrenar los dos modelos
modelo_t = LogisticRegression(max_iter=1000)
modelo_t.fit(train_t[FEATURES], train_t["responded_90d"])

modelo_c = LogisticRegression(max_iter=1000)
modelo_c.fit(train_c[FEATURES], train_c["responded_90d"])

# 3. Predecir probabilidades contrafactuales en test
test["p_tratado"] = modelo_t.predict_proba(test[FEATURES])[:, 1]
test["p_control"] = modelo_c.predict_proba(test[FEATURES])[:, 1]

# 4. Calcular uplift estimado
test["uplift_estimado"] = test["p_tratado"] - test["p_control"]

# 5. Funcion para calcular Qini
def calcular_qini(datos, col_score, steps=40):
    orden = datos.sort_values(col_score, ascending=False).reset_index(drop=True)
    es_t = (orden["treatment_group"] == "treatment").values
    resp = orden["responded_90d"].values

    n = len(orden)
    xs, ys = [0.0], [0.0]
    for i in np.linspace(1, n, steps).astype(int):
        sub_t = resp[:i][es_t[:i]]
        sub_c = resp[:i][~es_t[:i]]
        tasa_t = sub_t.mean() if len(sub_t) > 0 else 0
        tasa_c = sub_c.mean() if len(sub_c) > 0 else 0
        xs.append(i / n)
        ys.append((tasa_t - tasa_c) * n)

    xs, ys = np.array(xs), np.array(ys)
    area_modelo = np.trapezoid(ys, xs)
    area_azar = np.trapezoid(np.linspace(0, ys[-1], len(xs)), xs)
    return (area_modelo - area_azar) / n

qini_uplift = calcular_qini(test, "uplift_estimado")
qini_churn = calcular_qini(test, "prob_churn")

print(f"Qini T-Learner (Uplift): {qini_uplift:+.3f}")
print(f"Qini Churn (Riesgo):     {qini_churn:+.3f}")

Qini T-Learner (Uplift): +0.222
Qini Churn (Riesgo):     +0.006


In [6]:
# Evaluamos diferentes porcentajes de la base ordenados por uplift_estimado
porcentajes = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
filas = []

for pct in porcentajes:
    n_contactar = int(len(test) * pct)
    muestra = test.sort_values("uplift_estimado", ascending=False).head(n_contactar)

    vt = muestra.loc[muestra["treatment_group"] == "treatment", "value_90d"].mean()
    vc = muestra.loc[muestra["treatment_group"] == "control", "value_90d"].mean()
    uplift_prom = vt - vc
    valor_total = uplift_prom * n_contactar

    filas.append({
        "porcentaje_base": f"{int(pct*100)}%",
        "clientes": n_contactar,
        "uplift_por_cliente": round(uplift_prom, 2),
        "valor_incremental_total": round(valor_total, 2)
    })

tabla_politicas = pd.DataFrame(filas)
display(tabla_politicas)

# Ver que porcentaje da el maximo valor total
mejor_pct = tabla_politicas.loc[tabla_politicas["valor_incremental_total"].idxmax()]
print(f"El porcentaje optimo de contacto es {mejor_pct['porcentaje_base']} con un retorno de ${mejor_pct['valor_incremental_total']:,.2f}")

,porcentaje_base,clientes,uplift_por_cliente,valor_incremental_total
0,10%,420,11.62,4882.14
1,20%,840,9.59,8053.84
2,30%,1260,8.19,10318.67
3,40%,1680,7.15,12008.12
4,50%,2100,5.81,12208.29
5,60%,2520,4.53,11417.34
6,70%,2940,4.18,12290.16
7,80%,3360,3.80,12775.83
8,90%,3780,3.26,12305.59
9,100%,4200,2.84,11920.11


El porcentaje optimo de contacto es 80% con un retorno de $12,775.83


In [9]:
# 1. Correlacion entre lo que predijo el modelo y la verdad
corr_prob = test["uplift_estimado"].corr(test["true_uplift_prob"])
corr_val = test["uplift_estimado"].corr(test["true_uplift_value"])

print(f"Correlación con el uplift real (probabilidad): {corr_prob:.3f}")
print(f"Correlación con el uplift real (valor):        {corr_val:.3f}")

# 2. Promedio predicho vs real por cada cuadrante verdadero
resumen_cuadrantes = test.groupby("true_type").agg(
    clientes=("customer_id", "count"),
    uplift_estimado_prom=("uplift_estimado", "mean"),
    uplift_real_prob_prom=("true_uplift_prob", "mean"),
    uplift_real_valor_prom=("true_uplift_value", "mean")
).reset_index()

display(resumen_cuadrantes)

Correlación con el uplift real (probabilidad): 0.676
Correlación con el uplift real (valor):        0.666


,true_type,clientes,uplift_estimado_prom,uplift_real_prob_prom,uplift_real_valor_prom
0,lost_cause,1838,0.087713,0.02,0.18
1,persuadable,1010,0.289954,0.58,13.34
2,sleeping_dog,307,0.097619,-0.11,-4.84
3,sure_thing,1045,0.115469,0.03,0.72


In [8]:
print(test.columns.tolist())

['customer_id', 'treatment_group', 'recency_days', 'frequency', 'monetary', 'email_open_rate', 'responded_90d', 'value_90d', 'true_type', 'true_uplift_prob', 'true_uplift_value', 'prob_churn', 'p_tratado', 'p_control', 'uplift_estimado']


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Modelos para comparar dentro del T-learner
modelos = {
    "Logística (la del notebook)": LogisticRegression(max_iter=1000),
    "Árbol sin límite": DecisionTreeClassifier(random_state=42),
    "Árbol con max_depth=2": DecisionTreeClassifier(max_depth=2, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
}

filas = []

for nombre, m in modelos.items():
    # Entrenar modelo en tratados
    m_t = m.__class__(**m.get_params())
    m_t.fit(train_t[FEATURES], train_t["responded_90d"])

    # Entrenar modelo en control
    m_c = m.__class__(**m.get_params())
    m_c.fit(train_c[FEATURES], train_c["responded_90d"])

    # Predecir probabilidades y restar para sacar el uplift
    p_t = m_t.predict_proba(test[FEATURES])[:, 1]
    p_c = m_c.predict_proba(test[FEATURES])[:, 1]
    score = p_t - p_c

    # Correlacion con el uplift real de probabilidad
    corr = pd.Series(score).corr(test["true_uplift_prob"].reset_index(drop=True))

    # Calcular el indice Qini
    test_temp = test.copy()
    test_temp["score_temp"] = score
    qini_val = calcular_qini(test_temp, "score_temp")

    filas.append({
        "Modelo dentro del T-learner": nombre,
        "Correlación con el uplift real": round(corr, 3),
        "Qini": f"{qini_val:+.3f}"
    })

# Tabla final que pide la tarea
tabla_modelos = pd.DataFrame(filas)
display(tabla_modelos)

,Modelo dentro del T-learner,Correlación con el uplift real,Qini
0,Logística (la del notebook),0.676,+0.222
1,Árbol sin límite,0.353,+0.146
2,Árbol con max_depth=2,0.788,+0.248
3,Random Forest,0.899,+0.249
